<a href="https://colab.research.google.com/github/syakesaba/jupyter-notebooks/blob/main/github_copilot_sdk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Github Copilot SDK Python Sample Code
=====


# 初回実行必要

In [44]:
!pip install -qq github-copilot-sdk nest_asyncio pydantic httpx
import nest_asyncio
nest_asyncio.apply() # Google Colab自体がasyncio配下で動いているのでネストさせる。
from google.colab import userdata
GH_TOKEN=userdata.get("GH_TOKEN") # Secrets（🔑）からGH_TOKENをNotebook access可能にする
# GH_TOKEN=`gh auth token`

# Github Copilot Clientの起動

In [45]:
import asyncio
from copilot import CopilotClient, SubprocessConfig

client = CopilotClient(
    SubprocessConfig(
        use_logged_in_user=False,
        github_token=GH_TOKEN,
        log_level="debug",
    )
)

asyncio.run(client.start())

# 通常のプロンプト


## 最も基本的なプロンプト

In [46]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("Answer what 2+2 is."))

2 + 2 equals 4.


## システムプロンプト

In [47]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        system_message=SystemMessageReplaceConfig(content="あなたは真逆のことを言うエージェントです")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("多くのカラスは黒いですか？"))

はい、多くのカラスは黒いです。


# Tool

In [48]:
import asyncio

from copilot import CopilotSession, define_tool
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig

from pydantic import BaseModel, Field, StringConstraints

class Loto6Params(BaseModel):
    number: str = Field(description="Number for LOTO6", pattern=r'^\d{6}$')

# name must be: ^[a-zA-Z0-9_\\.-]+$
@define_tool(name="Loto6Tool", description="Loto6 check if win or not!", skip_permission=True)
async def _loto6(params: Loto6Params) -> str:
    return "WOW you got $10,000 !" if "123456" == params.number else "Sorry you got nothing!"

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        tools=[_loto6,],
        system_message=SystemMessageReplaceConfig(content="Use `Loto6Tool` if you receive 6-digits numbers and answer what you got.")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("My number is: 123456"))

You provided the number 123456 for Loto6. Let me check if this number is a winner.
Congratulations! Your number 123456 is a Loto6 winner—you've won $10,000!


# 画像認識

---



In [49]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

from pydantic import AnyUrl
import httpx
import base64

async def run(prompt: str, url: AnyUrl):
    async with httpx.AsyncClient() as http_client:
        image_response = await http_client.get(url)
        bin_image = image_response.content
        image_type = image_response.headers['content-type']
        b64_image = base64.b64encode(bin_image).decode('utf-8')
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1"
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(
        prompt,
        attachments=[
            {
                "type": "blob",
                "data": b64_image,
                "mimeType": image_type,
            }
        ]
    )
    await done.wait()
    await session.disconnect()

asyncio.run(run("これ何？", "http://www.sakado-jigenji.jp/images/k_logo.png"))

この画像は「慈眼寺（じげんじ）」というお寺のロゴやタイトル画像です。

上部には「真言宗 智山派 由城山」と書かれており、これは慈眼寺が「真言宗智山派」に属し、山号が「由城山」であることを示しています。左側のマークは寺院の紋章（寺紋）と思われます。

要約：
- 慈眼寺（じげんじ）というお寺
- 真言宗智山派
- 山号は由城山

何か詳細が必要であれば教えてください。


# Custom Agents & Sub-Agent Orchestration

# Skills

# AGENTS.md